# Goals
Check differences between the DataCite/Datacite and OpenAIRE/oai_datacite exports of datasets.

The notebook:
- Creates two testing datasets in Demo Dataverse:
  - A dataset with CC0 waiver
  - A dataset with "custom" terms
- Saves DataCite/Datacite and OpenAIRE/oai_datacite XML export files for both datasets, creating four XML files
- Creates two HTML files that show the diffeneces between:
  - The DataCite/Datacite and OpenAIRE/oai_datacite exports of the CC0 dataset
  - The DataCite/Datacite and OpenAIRE/oai_datacite exports of the custom terms dataset

# Import libraries

In [ ]:
import glob
import json
import os
from pathlib import Path
import platform
import requests
import sys
import time
from time import sleep

osName = platform.system()
if osName == 'Windows':
    dataverseFunctionsPath = os.path.join(
        'C:\\', 'Users', 'Owner', 'Documents', 'GitHub', 'dataverse-functions')
elif osName == 'Darwin':
    dataverseFunctionsPath = os.path.join(
        '/Users', 'juliangautier', 'dataverse-functions')
sys.path.append(dataverseFunctionsPath)  
from dataverse_functions import common_functions
from dataverse_functions import dataverse_functions

desktopPath = os.path.join(Path.home(), 'Desktop')

# Enter a user agent and your email address. Some Dataverse installations block requests from scripts.
# See https://www.whatismybrowser.com/detect/what-is-my-user-agent to get your user agent
userAgent = 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_14_6) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/81.0.4044.138 Safari/537.36'
emailAddress = 'juliangautier@g.harvard.edu'

headers = {
    'User-Agent': userAgent,
    'From': emailAddress}

# Get credentials

In [ ]:
filePath='/Users/juliangautier/Credentials/demo_creds.yaml'
credentials = common_functions.import_credentials(filePath=filePath)
installationUrl = credentials['installationURL']
apiKey = credentials['apiToken']

# Create testing datasets

## Create CC0 dataset

In [ ]:
# Save "testing_json_file" to testingDataverseJsonFilePath variable
testingDataverseJsonFilePath = '/Users/juliangautier/dataverse-functions/dataverse-scripts/create_dataset/all_metadata_fields.json'
with open(testingDataverseJsonFilePath, 'r') as f:
    dataverseJsonExortString = f.read()
    dataverseJsonExortDictCC0 = json.loads(dataverseJsonExortString)

# Make sure license is CC0
dataverseJsonExortDictCC0['datasetVersion']['license'] = {
    "name": "CC0 1.0",
    "uri": "http://creativecommons.org/publicdomain/zero/1.0"
}

## Change title of test dataset to "CC0 dataset"
for field in dataverseJsonExortDictCC0['datasetVersion']['metadataBlocks']['citation']['fields']:
    if field['typeName'] == 'title':
        field['value'] = 'CC0 dataset'

dataverseAlias = 'juliangautier'
messageDict = dataverse_functions.create_testing_dataset(
    installationUrl, dataverseAlias='juliangautier', 
    apiKey=apiKey, headers={}, validateMetadata=True,
    jsonDataDict=dataverseJsonExortDictCC0)

cc0DatasetPid = messageDict['data']['persistentId']
print(cc0DatasetPid)

## Create custom terms dataset

In [ ]:
# Save "testing_json_file" to testingDataverseJsonFilePath variable
testingDataverseJsonFilePath = '/Users/juliangautier/dataverse-functions/dataverse-scripts/create_dataset/all_metadata_fields.json'
with open(testingDataverseJsonFilePath, 'r') as f:
    dataverseJsonExortString = f.read()
    dataverseJsonExortDictCustomTerms = json.loads(dataverseJsonExortString)

# Make sure license is "Custom"
customTermsFieldsList = [
    'termsOfUse',
    'confidentialityDeclaration',
    'specialPermissions',
    'restrictions',
    'citationRequirements',
    'depositorRequirements',
    'conditions',
    'disclaimer',
    'dataAccessPlace',
    'originalArchive',
    'availabilityStatus',
    'contactForAccess',
    'sizeOfCollection',
    'studyCompletion'
]

for field in customTermsFieldsList:
    dataverseJsonExortDictCustomTerms['datasetVersion'][field] = field

# Change title of test dataset to "Custom terms dataset"
for field in dataverseJsonExortDictCustomTerms['datasetVersion']['metadataBlocks']['citation']['fields']:
    if field['typeName'] == 'title':
        field['value'] = 'Custom terms dataset'

dataverseAlias = 'juliangautier'
messageDict = dataverse_functions.create_testing_dataset(
    installationUrl, dataverseAlias='juliangautier', 
    apiKey=apiKey, headers={}, validateMetadata=True,
    jsonDataDict=dataverseJsonExortDictCustomTerms)

customTermsDatasetPid = messageDict['data']['persistentId']
print(customTermsDatasetPid)

# Save Datacite and oai_datacite exports

In [ ]:
# cc0DatasetPid = 'doi:10.70122/FK2/MNOMT7'
# customTermsDatasetPid = 'doi:10.70122/FK2/XRTVOS'

In [ ]:
# Create directoryPath
currentTime = time.strftime('%Y.%m.%d_%H.%M.%S')
directoryPath = os.path.join(f'diffs and exports - {currentTime}')
os.makedirs(directoryPath)

# Get suffixes of both exports for renaming the files later
cc0DatasetPidSuffix = cc0DatasetPid.split('/')[-1]
customTermsDatasetPidSuffix = customTermsDatasetPid.split('/')[-1]

## Create and rename exports

In [ ]:
# Create and rename oai_datacite exports

downloadStatusFilePath = os.path.join(
    directoryPath, 
    f'oai_datacite - downloadStatus - {currentTime}.csv')

session = requests.Session()
dataverse_functions.save_dataset_exports(
    directoryPath, downloadStatusFilePath, installationUrl, 
    datasetPidList=[cc0DatasetPid],
    exportFormat='oai_datacite', session=session, 
    timeout=60, verify=False, excludeFiles=False, 
    sleepDuration=0.75, version=':latest', headers={}, apiKey=apiKey)

sleep(1)

dataverse_functions.save_dataset_exports(
    directoryPath, downloadStatusFilePath, installationUrl, 
    datasetPidList=[customTermsDatasetPid],
    exportFormat='oai_datacite', session=session, 
    timeout=60, verify=False, excludeFiles=False, 
    sleepDuration=0.75, version=':latest', headers={}, apiKey=apiKey)

# Rename files
for oldFilePath in glob.glob(os.path.join(directoryPath, '*.xml')):
    fileName = os.path.basename(oldFilePath)
    if cc0DatasetPidSuffix in fileName:
        newFileName = f'oai_datacite export - CC0 - {cc0DatasetPidSuffix}.xml'
        cc0_oai_datacite_file_path = os.path.join(directoryPath, newFileName)
        os.rename(oldFilePath, cc0_oai_datacite_file_path)
    elif customTermsDatasetPidSuffix in fileName:
        newFileName = f'oai_datacite export - custom terms - {customTermsDatasetPidSuffix}.xml'
        custom_terms_oai_datacite_file_path = os.path.join(directoryPath, newFileName)
        os.rename(oldFilePath, custom_terms_oai_datacite_file_path)

os.remove(downloadStatusFilePath)

# Create and rename Datacite exports

downloadStatusFilePath = os.path.join(directoryPath, f'Datacite - downloadStatus - {currentTime}.csv')

sleep(1)
dataverse_functions.save_dataset_exports(
    directoryPath, downloadStatusFilePath, installationUrl, 
    datasetPidList=[cc0DatasetPid],
    exportFormat='Datacite', session=session, 
    timeout=60, verify=False, excludeFiles=False, 
    sleepDuration=0.75, version=':latest', headers={}, apiKey=apiKey)

sleep(1)

dataverse_functions.save_dataset_exports(
    directoryPath, downloadStatusFilePath, installationUrl, 
    datasetPidList=[customTermsDatasetPid],
    exportFormat='Datacite', session=session, 
    timeout=60, verify=False, excludeFiles=False, 
    sleepDuration=0.75, version=':latest', headers={}, apiKey=apiKey)

session.close()

# Rename files
for oldFilePath in glob.glob(os.path.join(directoryPath, '*.xml')):
    fileName = os.path.basename(oldFilePath)
    if cc0DatasetPidSuffix in fileName and 'oai_datacite export' not in fileName:
        newFileName = f'Datacite export - CC0 - {cc0DatasetPidSuffix}.xml'
        cc0_Datacite_file_path = os.path.join(directoryPath, newFileName)
        os.rename(oldFilePath, cc0_Datacite_file_path)
    elif customTermsDatasetPidSuffix in fileName and 'oai_datacite export' not in fileName:
        newFileName = f'Datacite export - custom terms - {customTermsDatasetPidSuffix}.xml'
        custom_terms_Datacite_file_path = os.path.join(directoryPath, newFileName)
        os.rename(oldFilePath, custom_terms_Datacite_file_path)

os.remove(downloadStatusFilePath)

# Create two diffs
- 1 for exports of the CC0 dataset: cc0_oai_datacite_file_path and cc0_Datacite_file_path
- 1 for exports of the custom terms dataset: custom_terms_oai_datacite_file_path and custom_terms_Datacite_file_path

In [ ]:
# Create diff file for CC0 datasets
oldDiffFilePath = common_functions.create_diff_html_of_xml_files(
    directoryPath,
    cc0_oai_datacite_file_path, 
    cc0_Datacite_file_path)

# Rename diff file
newDiffFileName = f'diff_CC0_datasets.html'
newDiffFilePath = os.path.join(directoryPath, newDiffFileName)
os.rename(oldDiffFilePath, newDiffFilePath)

# Create diff for custom terms datasets
oldDiffFilePath = common_functions.create_diff_html_of_xml_files(
    directoryPath,
    custom_terms_oai_datacite_file_path, 
    custom_terms_Datacite_file_path)

# Rename diff file
newDiffFileName = f'diff_custom_terms_datasets.html'
newDiffFilePath = os.path.join(directoryPath, newDiffFileName)
os.rename(oldDiffFilePath, newDiffFilePath)